# Used Car Price Prediction
## Notebook 01 — Data Audit & Exploratory Data Analysis

**Goal:** Understand the raw dataset, identify data quality issues, and document findings before any cleaning or modeling.

**Input:** `cardekho_dataset.csv`

**Outputs:**
- Tables in `outputs/tables/`
- Figures in `outputs/figures/`
- Report in `outputs/reports/`

**Notebook roadmap:**
1. Environment setup
2. Load raw data
3. Dataset overview
4. Missing values and duplicates
5. Validity checks
6. IQR outlier screening
7. Target variable analysis
8. Numerical features
9. Categorical features
10. Features vs target
11. Correlation analysis
12. Extreme records
13. Summary report

## 1. Environment setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'cardekho_dataset.csv'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'

for d in (TABLE_DIR, FIGURE_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 300

TARGET = 'selling_price'
TOP_N = 15
NUMERICAL_COLUMNS = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', TARGET]

def save_fig(name):
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / name, dpi=300, bbox_inches='tight')
    plt.show()

print('Data path:', DATA_PATH)
print('Exists   :', DATA_PATH.exists())

## 2. Load raw data

The stored index column (`Unnamed: 0`) is dropped immediately because it carries no information.

In [ ]:
raw = pd.read_csv(DATA_PATH)
unnamed = [c for c in raw.columns if str(c).lower().startswith('unnamed:')]
raw = raw.drop(columns=unnamed)
print(f'Rows: {len(raw):,} | Columns: {raw.shape[1]:,}')
print('Dropped columns:', unnamed)
raw.head()

## 3. Dataset overview

Inspect shape, data types, and unique value counts per column.

In [ ]:
print('Shape:', raw.shape)
raw.info()

In [ ]:
unique_summary = (
    pd.DataFrame({'unique_values': raw.nunique(), 'dtype': raw.dtypes.astype(str)})
    .sort_values('unique_values', ascending=False)
)
unique_summary.to_csv(TABLE_DIR / 'unique_values_summary.csv')
unique_summary

In [ ]:
raw.describe(include='all').T.to_csv(TABLE_DIR / 'dataset_summary.csv')
raw.describe(include='all').T

## 4. Missing values and duplicates

In [ ]:
missing = pd.DataFrame({
    'missing_count': raw.isna().sum(),
    'missing_percent': (raw.isna().mean() * 100).round(2),
}).sort_values('missing_percent', ascending=False)
missing.index.name = 'feature'
missing.to_csv(TABLE_DIR / 'missing_values.csv')
missing[missing['missing_count'] > 0]

In [ ]:
dup_count = int(raw.duplicated().sum())
print(f'Duplicate rows: {dup_count:,} ({dup_count / len(raw) * 100:.2f}%)')
raw[raw.duplicated(keep=False)].to_csv(TABLE_DIR / 'duplicate_records.csv', index=False)
raw[raw.duplicated(keep=False)].head(20)

## 5. Validity checks

Business-rule checks for impossible values. These are **flagged**, not removed yet.

In [ ]:
rules = {
    'negative_vehicle_age': raw['vehicle_age'] < 0,
    'negative_km_driven': raw['km_driven'] < 0,
    'nonpositive_mileage': raw['mileage'] <= 0,
    'nonpositive_engine': raw['engine'] <= 0,
    'nonpositive_max_power': raw['max_power'] <= 0,
    'seats_outside_2_10': ~raw['seats'].between(2, 10),
    'nonpositive_selling_price': raw[TARGET] <= 0,
}
validity = pd.DataFrame([
    {'rule': k, 'flagged_count': int(v.sum()), 'flagged_percent': round(v.mean() * 100, 2)}
    for k, v in rules.items()
])
validity.to_csv(TABLE_DIR / 'data_validity_summary.csv', index=False)
validity

In [ ]:
flagged_mask = pd.Series(False, index=raw.index)
for v in rules.values():
    flagged_mask |= v
flagged = raw[flagged_mask]
flagged.to_csv(TABLE_DIR / 'potentially_invalid_records.csv', index=False)
print(f'Total flagged rows: {len(flagged):,}')
flagged.head(20)

## 6. IQR outlier screening

Outliers are reported for every numerical column. They are **not** removed here.

In [ ]:
rows = []
for col in NUMERICAL_COLUMNS:
    s = raw[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < lower) | (s > upper)
    rows.append({
        'feature': col, 'q1': q1, 'q3': q3, 'iqr': iqr,
        'lower_bound': lower, 'upper_bound': upper,
        'outlier_count': int(mask.sum()),
        'outlier_percent': round(mask.mean() * 100, 2),
    })
outliers = pd.DataFrame(rows)
outliers.to_csv(TABLE_DIR / 'numerical_outlier_summary.csv', index=False)
outliers

## 7. Target variable analysis

Distribution, skewness, and log-transform check.

In [ ]:
target = raw[TARGET].dropna()
stats = target.describe().to_frame().T
stats['median'] = target.median()
stats['skewness'] = target.skew()
stats['kurtosis'] = target.kurtosis()
stats.to_csv(TABLE_DIR / 'target_statistics.csv', index=False)
stats

In [ ]:
positive = target[target > 0]
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(target, bins=50, kde=True, ax=axes[0]); axes[0].set_title('Selling Price')
sns.boxplot(x=target, ax=axes[1]); axes[1].set_title('Boxplot')
sns.histplot(np.log1p(positive), bins=50, kde=True, ax=axes[2]); axes[2].set_title('log(1 + price)')
save_fig('target_distribution.png')

## 8. Numerical features

Distribution and boxplot for each numeric feature.

In [ ]:
num_cols = [c for c in raw.select_dtypes(include=np.number).columns if c != TARGET]
num_summary = raw[num_cols].describe().T
num_summary['skewness'] = raw[num_cols].skew()
num_summary['kurtosis'] = raw[num_cols].kurtosis()
num_summary.to_csv(TABLE_DIR / 'numerical_features_summary.csv')
num_summary

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    sns.histplot(raw[col].dropna(), bins=40, kde=True, ax=axes[0]); axes[0].set_title(f'{col} distribution')
    sns.boxplot(x=raw[col], ax=axes[1]); axes[1].set_title(f'{col} boxplot')
    save_fig(f'numerical_{col}.png')

## 9. Categorical features

Top categories per feature.

In [ ]:
cat_cols = raw.select_dtypes(include=['object', 'str', 'category']).columns.tolist()
summary = []
for col in cat_cols:
    vc = raw[col].value_counts(dropna=False)
    summary.append({'feature': col, 'unique_values': raw[col].nunique(),
                    'most_frequent': vc.index[0] if len(vc) else np.nan,
                    'top_count': int(vc.iloc[0]) if len(vc) else 0})
    if col in ('car_name', 'model'):
        continue
    top = vc.head(TOP_N)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top.values, y=top.index, orient='h')
    plt.title(f'Top {TOP_N} categories: {col}')
    save_fig(f'categorical_{col}.png')
pd.DataFrame(summary).to_csv(TABLE_DIR / 'categorical_features_summary.csv', index=False)
pd.DataFrame(summary)

## 10. Features vs target

Scatter plots for numeric features and boxplots for key categorical features.

In [ ]:
for col in num_cols:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=raw, x=col, y=TARGET, alpha=0.35, s=20)
    plt.title(f'{col} vs {TARGET}')
    save_fig(f'price_vs_{col}.png')

In [ ]:
for col in ['brand', 'fuel_type', 'seller_type', 'transmission_type']:
    top = raw[col].value_counts().head(TOP_N).index
    plt.figure(figsize=(11, 5))
    sns.boxplot(data=raw[raw[col].isin(top)], x=col, y=TARGET, order=top)
    plt.xticks(rotation=45, ha='right')
    plt.title(f'{TARGET} by {col}')
    save_fig(f'price_by_{col}.png')

## 11. Correlation analysis

In [ ]:
corr = raw.select_dtypes(include=np.number).corr()
corr.to_csv(TABLE_DIR / 'correlation_matrix.csv')

target_corr = corr[TARGET].drop(TARGET).sort_values(ascending=False, key=abs)
target_corr.to_csv(TABLE_DIR / 'target_correlations.csv', header=['correlation'])
target_corr

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.4)
plt.title('Correlation matrix')
save_fig('correlation_matrix.png')

## 12. Extreme records

Top 10 rows per numerical column and seat distribution.

In [ ]:
for col in ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', TARGET]:
    print(f'\nTop 10 by {col}:')
    display(raw.nlargest(10, col))
    raw.nlargest(10, col).to_csv(TABLE_DIR / f'top_10_{col}.csv', index=False)

In [ ]:
seat_dist = raw['seats'].value_counts(dropna=False).sort_index()
seat_dist.to_csv(TABLE_DIR / 'seat_distribution.csv', header=['count'])
seat_dist

## 13. Summary report

Key findings that will drive the cleaning decisions in the next notebook.

In [ ]:
report = {
    'rows': len(raw),
    'columns': raw.shape[1],
    'total_missing': int(raw.isna().sum().sum()),
    'exact_duplicates': int(raw.duplicated().sum()),
    'target_mean': float(raw[TARGET].mean()),
    'target_median': float(raw[TARGET].median()),
    'target_std': float(raw[TARGET].std()),
    'target_skewness': float(raw[TARGET].skew()),
}
pd.Series(report).to_csv(REPORT_DIR / 'eda_summary.csv')

lines = ['EXPLORATORY DATA ANALYSIS REPORT', '=' * 45]
lines += [f'{k}: {v}' for k, v in report.items()]
text = '\n'.join(lines)
(REPORT_DIR / 'eda_summary.txt').write_text(text, encoding='utf-8')
print(text)

### Findings

| Issue | Evidence | Action in next notebook |
|---|---|---|
| Stored index column | `Unnamed: 0` present | Dropped on load |
| Rows with `seats = 0` | e.g. Honda City, Nissan Kicks | Filter to `2 <= seats <= 10` |
| `vehicle_age = 0` | A few near-new cars | Kept; used for `km_per_year` guard |
| Extreme `km_driven` | e.g. 425,785 km Alto | Flagged by IQR; not dropped |
| Extreme `selling_price` | e.g. 39.5M Ferrari | Flagged by IQR; not dropped |
| Inconsistent model names | `RediGO` vs `redi-GO`, `Dzire VXI` variants | Normalized in `preprocessing.py` |
| Rare categories | Luxury/exotic brands | Grouped into `__rare__` |
| Non-positive numeric values | mileage, engine, max_power | Filtered out |
| Duplicates | Exact duplicate rows | Removed before split |

**Decision:** No rows are removed in this notebook. Cleaning is performed in `02_preprocessing.ipynb` before the train/test split.